In [1]:
import pandas as pd

In [2]:
# cleaned and standardized files

df_listings_data = pd.read_csv("../data/London/cleaned/listingsdata_clean.csv")
df_calendar = pd.read_csv("../data/London/cleaned/calendar_clean.csv")
df_reviews_data = pd.read_csv("../data/London/cleaned/reviewsdata_clean.csv")
df_listings_info = pd.read_csv("../data/London/cleaned/listingsinfo_clean.csv")
df_reviews_info = pd.read_csv("../data/London/cleaned/reviewsinfo_clean.csv")
df_neighbourhoods = pd.read_csv("../data/London/cleaned/neighbourhoods_clean.csv")

In [3]:
# review summary

review_summary = (
    df_reviews_data
    .groupby("listing_id")
    .agg(
        total_reviews=("id", "count"),
        first_review_date=("date", "min"),
        latest_review_date=("date", "max")
    )
    .reset_index()
)

In [4]:
review_summary

,listing_id,total_reviews,first_review_date,latest_review_date
0,13913,55,2010-08-18,2025-08-21
1,15400,97,2009-12-21,2025-04-05
2,17402,56,2011-03-21,2024-02-19
3,24328,95,2010-11-15,2025-07-05
4,36274,15,2011-03-20,2025-09-06
...,...,...,...,...
72744,1507145680187854643,1,2025-09-13,2025-09-13
72745,1507215411373476327,1,2025-09-13,2025-09-13
72746,1507880654074076794,1,2025-09-13,2025-09-13
72747,1508080538383464858,1,2025-09-12,2025-09-12


In [5]:
# joining to listings

listing_master = df_listings_data.merge(
    review_summary,
    left_on="id",
    right_on="listing_id",
    how="left"
)


In [6]:
df_calendar["available"].value_counts() #quick check

available
f    21313866
t    14044108
Name: count, dtype: int64

In [7]:
# calculation occupancy

calendar_metrics = (
    df_calendar
    .assign(
        booked=lambda x: x["available"].eq("f")
    )
    .groupby("listing_id")
    .agg(
        total_days=("date", "count"),
        booked_days=("booked", "sum")
    )
    .reset_index()
)

In [8]:
# occupancy rate

calendar_metrics["occupancy_rate"] = (
    calendar_metrics["booked_days"]
    / calendar_metrics["total_days"]
) * 100

In [9]:
listing_master = df_listings_data.merge(
    calendar_metrics,
    left_on="id",
    right_on="listing_id",
    how="left"
)

In [10]:
# revenue estimation

listing_master["estimated_revenue"] = (
    listing_master["booked_days"]
    *
    listing_master["price_clean"]
)

In [11]:
# Neighbourhood Enrichment

neighbourhood_metrics = (
    listing_master
    .groupby("neighbourhood_cleansed")
    .agg(
        neighbourhood_median_price=(
            "price_clean",
            "median"
        ),
        neighbourhood_listing_count=(
            "id",
            "count"
        ),
        neighbourhood_avg_rating=(
            "review_scores_rating",
            "mean"
        )
    )
    .reset_index()
)

In [12]:
listing_master = listing_master.merge(
    neighbourhood_metrics,
    on="neighbourhood_cleansed",
    how="left"
)

In [13]:
listing_master["host_since"].dtype  # quick datatype check

<StringDtype(storage='python', na_value=nan)>

In [14]:
listing_master["host_since"] = pd.to_datetime(
    listing_master["host_since"],
    errors="coerce"
)

In [15]:
listing_master["first_review"] = pd.to_datetime(
    listing_master["first_review"],
    errors="coerce"
)

In [16]:
# deriving features

# host tenure

listing_master["host_tenure_years"] = (
    pd.Timestamp.today()
    - listing_master["host_since"]
).dt.days / 365.25

listing_master["host_tenure_years"] = (
    listing_master["host_tenure_years"]
    .round(2)
)

In [17]:
# reviews per year

listing_master["listing_age_years"] = (
    pd.Timestamp.today()
    - listing_master["first_review"]
).dt.days / 365.25

In [18]:
listing_master["review_frequency"] = (
    listing_master["number_of_reviews"]
    / listing_master["listing_age_years"]
)

In [19]:
# price per bedroom

listing_master["price_per_bedroom"] = (
    listing_master["price_clean"]
    /
    listing_master["bedrooms"].replace(0, pd.NA)
)

In [20]:
# master table

listing_master.shape

(96871, 92)

In [21]:
listing_master.columns

Index(['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name',
       'description', 'neighborhood_overview', 'picture_url', 'host_id',
       'host_url', 'host_name', 'host_since', 'host_location', 'host_about',
       'host_response_time', 'host_response_rate', 'host_acceptance_rate',
       'host_is_superhost', 'host_thumbnail_url', 'host_picture_url',
       'host_neighbourhood', 'host_listings_count',
       'host_total_listings_count', 'host_verifications',
       'host_has_profile_pic', 'host_identity_verified', 'neighbourhood',
       'neighbourhood_cleansed', 'latitude', 'longitude', 'property_type',
       'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms',
       'beds', 'amenities', 'price', 'minimum_nights', 'maximum_nights',
       'minimum_minimum_nights', 'maximum_minimum_nights',
       'minimum_maximum_nights', 'maximum_maximum_nights',
       'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'has_availability',
       'availability_3

In [22]:
from pathlib import Path

Path("../data/London/enriched").mkdir(
    parents=True,
    exist_ok=True
)

listing_master.to_csv(
    "../data/London/enriched/listing_master.csv",
    index=False
)